# Clean X

H4 total tariff change, 2023 minus 2022 (ct/kWh).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "Data" / "raw"
SUPPORT_DIR = PROJECT_DIR / "supporting_inputs"
OUTPUT_DIR = PROJECT_DIR / "cleaning_outputs_simple"
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
print(RAW_DIR.exists())
print(SUPPORT_DIR.exists())
print(OUTPUT_DIR.exists())

True
True
True


## Municipality identifiers

In [4]:
municipality_changes = pd.read_csv(SUPPORT_DIR / "bfs_commune_mutations.csv")
municipalities = pd.read_csv(SUPPORT_DIR / "bfs_levels_2026.csv")
municipalities = municipalities[["BfsCode", "Name", "CantonId", "Canton"]]
municipalities = municipalities.rename(columns={
    "BfsCode": "bfs",
    "Name": "commune",
    "CantonId": "canton_id",
    "Canton": "canton"
})
assert municipalities["bfs"].is_unique
valid_ids = set(municipalities["bfs"])
print("2026 municipalities:", len(municipalities))

2026 municipalities: 2110


In [15]:
display(municipalities.head())
display(municipality_changes.head())

,bfs,commune,canton_id,canton
0,6158,Vionnaz,23,Valais / Wallis
1,3023,Speicher,15,Appenzell Ausserrhoden
2,6011,Zwischbergen,23,Valais / Wallis
3,2228,Villars-sur-Glâne,10,Fribourg / Freiburg
4,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg


,MutationNumber,MutationDate,InitialHistoricalCode,InitialCode,InitialName,InitialParentHistoricalCode,InitialParentName,InitialStep,TerminalHistoricalCode,TerminalCode,TerminalName,TerminalParentHistoricalCode,TerminalParentName,TerminalStep
0,2773,01.01.2010,10948,343,Untersteckholz,10072,Amtsbezirk Aarwangen,29,14975,329,Langenthal,10286,Verwaltungskreis Oberaargau,26
1,2773,01.01.2010,12627,329,Langenthal,10072,Amtsbezirk Aarwangen,26,14975,329,Langenthal,10286,Verwaltungskreis Oberaargau,26
2,2774,01.01.2010,13366,531,Ballmoos,10216,Amtsbezirk Fraubrunnen,29,14976,540,Jegenstorf,10288,Verwaltungskreis Bern-Mittelland,26
3,2774,01.01.2010,13375,540,Jegenstorf,10216,Amtsbezirk Fraubrunnen,26,14976,540,Jegenstorf,10288,Verwaltungskreis Bern-Mittelland,26
4,2775,01.01.2010,13423,601,Aeschlen,10213,Amtsbezirk Konolfingen,29,14977,619,Oberdiessbach,10288,Verwaltungskreis Bern-Mittelland,26


In [5]:
changed_identifier = municipality_changes["InitialCode"] != municipality_changes["TerminalCode"]
merger_links = municipality_changes.loc[changed_identifier]
merger_links = merger_links[["InitialCode", "TerminalCode"]]
merger_links = merger_links.drop_duplicates()
assert merger_links["InitialCode"].is_unique, "Ambiguous municipality merger links."
next_identifier = dict(zip(merger_links["InitialCode"], merger_links["TerminalCode"]))

In [6]:
def to_2026(source_identifier):
    if pd.isna(source_identifier):
        return np.nan
    current_identifier = int(source_identifier)
    visited_identifiers = set()
    while current_identifier in next_identifier:
        assert current_identifier not in visited_identifiers, "Cycle in municipality merger links."
        visited_identifiers.add(current_identifier)
        destination_identifier = next_identifier[current_identifier]
        current_identifier = int(destination_identifier)
    return current_identifier

In [16]:
display(merger_links.head())
print(next_identifier[343])

,InitialCode,TerminalCode
0,343,329
2,531,540
4,601,619
6,752,756
7,753,756


329


## Select H4 tariffs

In [7]:
tariffs = pd.read_csv(RAW_DIR / "elcom_tariffs_H2_H4_H7.csv")
is_h4 = tariffs["category"] == "H4"
is_selected_year = tariffs["period"].isin([2022, 2023])
selected_tariffs = tariffs.loc[is_h4 & is_selected_year]
selected_tariffs = selected_tariffs.copy()
selected_tariffs["bfs"] = selected_tariffs["municipality"].map(to_2026)
print("Selected H4 records:", len(selected_tariffs))

Selected H4 records: 4687


In [17]:
display(selected_tariffs[
    ["municipality", "bfs", "operator", "period", "total"]
].head())

,municipality,bfs,operator,period,total
28528,4319,4319,280,2022,20.316
28529,4322,4324,280,2022,19.916
28530,404,404,636,2022,23.286
28532,3603,3603,417,2022,15.433
30734,3295,3295,455,2022,21.677


## Conflicting prices

Exclude the entire mapped municipality if one source municipality/operator/year has conflicting totals.

In [18]:
price_keys = ["municipality", "operator", "period"]
price_groups = selected_tariffs.groupby(price_keys)
distinct_prices = price_groups["total"].nunique()
is_conflict = distinct_prices > 1
price_conflicts = distinct_prices.loc[is_conflict]
price_conflicts = price_conflicts.reset_index(name="distinct_totals")
conflict_destinations = price_conflicts["municipality"].map(to_2026)
conflicting_ids = set(conflict_destinations)
display(price_conflicts)

,municipality,operator,period,distinct_totals
0,6459,691,2023,2


## Identical duplicates

In [9]:
duplicate_keys = ["municipality", "operator", "period", "total"]
unique_tariffs = selected_tariffs.drop_duplicates(duplicate_keys)
removed_duplicates = len(selected_tariffs) - len(unique_tariffs)
print("Identical price duplicates removed:", removed_duplicates)
is_known_municipality = unique_tariffs["bfs"].isin(valid_ids)
has_price_conflict = unique_tariffs["bfs"].isin(conflicting_ids)
valid_tariffs = unique_tariffs.loc[is_known_municipality & ~has_price_conflict]
valid_tariffs = valid_tariffs.copy()

Identical price duplicates removed: 2


In [19]:
print("Before removing duplicates:", len(selected_tariffs))
print("After removing duplicates:", len(unique_tariffs))
print("After excluding invalid/conflicting municipalities:", len(valid_tariffs))

Before removing duplicates: 4687
After removing duplicates: 4685
After excluding invalid/conflicting municipalities: 4676


## Average within each operator

In [10]:
operator_keys = ["bfs", "operator", "period"]
operator_groups = valid_tariffs.groupby(operator_keys, as_index=False)
operator_tariffs = operator_groups.agg(
    total=("total", "mean"), predecessor_count=("municipality", "nunique"))
duplicate_operators = operator_tariffs.duplicated(operator_keys)
assert not duplicate_operators.any()
print("Municipality/operator/year records:", len(operator_tariffs))

Municipality/operator/year records: 4601


In [20]:
display(operator_tariffs.head())

,bfs,operator,period,total,predecessor_count
0,1,486,2022,17.084,1
1,1,486,2023,21.878,1
2,2,486,2022,17.084,1
3,2,486,2023,21.878,1
4,3,486,2022,17.084,1


## Average operators equally

In [11]:
annual_groups = operator_tariffs.groupby(["bfs", "period"], as_index=False)
annual_tariffs = annual_groups.agg(
    tariff=("total", "mean"), n_operators=("operator", "nunique"))
duplicate_years = annual_tariffs.duplicated(["bfs", "period"])
assert not duplicate_years.any()
display(annual_tariffs.head())

,bfs,period,tariff,n_operators
0,1,2022,17.084,1
1,1,2023,21.878,1
2,2,2022,17.084,1
3,2,2023,21.878,1
4,3,2022,17.084,1


In [21]:
has_multiple_operators = annual_tariffs["n_operators"] > 1
display(annual_tariffs.loc[has_multiple_operators].head())

,bfs,period,tariff,n_operators
78,58,2022,19.5925,2
79,58,2023,31.3245,2
130,92,2022,18.2990,2
131,92,2023,24.7785,2
154,112,2022,16.3025,2


## Calculate X

In [12]:
tariffs_by_year = annual_tariffs.pivot(index="bfs", columns="period", values="tariff")
tariffs_by_year = tariffs_by_year.rename(columns={2022: "tariff_2022", 2023: "tariff_2023"})
municipality_names = municipalities[["bfs", "commune"]]
clean_x = municipality_names.merge(tariffs_by_year, on="bfs", how="left", validate="1:1")
clean_x["X"] = clean_x["tariff_2023"] - clean_x["tariff_2022"]
clean_x["tariff_conflict"] = clean_x["bfs"].isin(conflicting_ids)
clean_x["X_available"] = clean_x["X"].notna()

In [22]:
is_municipality_58 = clean_x["bfs"] == 58
display(clean_x.loc[is_municipality_58])

,bfs,commune,tariff_2022,tariff_2023,X,tariff_conflict,X_available
368,58,Glattfelden,19.5925,31.3245,11.732,False,True


## Check and save

In [13]:
assert clean_x["bfs"].is_unique
assert len(clean_x) == len(municipalities)
conflicting_values = clean_x.loc[clean_x["tariff_conflict"], "X"]
assert conflicting_values.isna().all()
available_2022 = clean_x["tariff_2022"].notna()
available_2023 = clean_x["tariff_2023"].notna()
available_both_years = available_2022 & available_2023
assert clean_x["X_available"].equals(available_both_years)
print("Valid X:", clean_x["X_available"].sum())
print("Missing X:", clean_x["X"].isna().sum())
display(clean_x.head(10))

Valid X: 2097
Missing X: 13


,bfs,commune,tariff_2022,tariff_2023,X,tariff_conflict,X_available
0,6158,Vionnaz,20.894,31.937,11.043,False,True
1,3023,Speicher,20.283,26.723,6.440,False,True
2,6011,Zwischbergen,8.193,8.493,0.300,False,True
3,2228,Villars-sur-Glâne,21.157,25.352,4.195,False,True
4,2230,Villarsel-sur-Marly,21.157,25.352,4.195,False,True
5,181,Wila,17.084,21.878,4.794,False,True
6,71,Wil (ZH),17.084,21.878,4.794,False,True
7,4045,Wettingen,19.535,31.774,12.239,False,True
8,121,Wetzikon (ZH),20.215,29.262,9.047,False,True
9,4081,Widen,19.189,19.663,0.474,False,True


In [23]:
clean_x.to_csv(OUTPUT_DIR / "clean_X.csv", index=False)
print("Saved: cleaning_outputs_simple/clean_X.csv")

Saved: cleaning_outputs_simple/clean_X.csv
